In [ ]:
import json
from datasets import load_dataset
from transformers import pipeline
import pandas as pd
import time
import pronto
import re
from nltk.metrics import jaccard_distance

In [ ]:
raw_data = load_dataset("disi-unibo-nlp/ReMedQA")["medqa_mcq"]
print(json.dumps(raw_data[0], indent=2))
orig_data = pd.DataFrame({"text": raw_data["question"], "prompt": raw_data["prompt"], "gold_answer": raw_data["answer"]})
orig_data.reset_index(inplace=True)
orig_data.head()

In [ ]:
disease_ner_pipeline = pipeline(
    model="OpenMed/OpenMed-NER-PathologyDetect-PubMed-109M",
    aggregation_strategy="simple"
)
disease_ner_pipeline.tokenizer.model_max_length = 512

start = time.time()
results = disease_ner_pipeline(orig_data["text"].tolist(), 
                                batch_size=16,
                                stride=50)
print("NER pipeline took {:.2f} seconds".format(time.time() - start))
print(results)

In [ ]:
def merge_adjacent_entities(entities, text):
    if not entities:
        return entities
    entities = sorted(entities, key=lambda e: e["start"])
    merged = [dict(entities[0])]
    for entity in entities[1:]:
        prev = merged[-1]
        gap = text[prev["end"]:entity["start"]]
        if entity["entity_group"] == prev["entity_group"] and gap.strip() == "":
            prev["end"] = entity["end"]
            prev["word"] = text[prev["start"]:prev["end"]]
            prev["score"] = min(prev["score"], entity["score"])
        else:
            merged.append(dict(entity))
    return merged

unique_entities = set()
total_entities = 0
for i, entities in enumerate(results):
    entities = merge_adjacent_entities(entities, orig_data.loc[i, "text"])
    disease_mentions = [orig_data.loc[i, "text"][e["start"]:e["end"]] for e in entities]
    orig_data.loc[i, "disease_mentions"] = json.dumps(disease_mentions)
    unique_entities.update(disease_mentions)
    total_entities += len(disease_mentions)
orig_data.to_json("data/medqa_mcq_with_disease_mentions.json", orient="records", lines=True, force_ascii=False)

print(f"Total disease mentions: {total_entities}")
print(f"Unique disease mentions: {len(unique_entities)}")

In [ ]:
#find synonyms for each disease mention by searching through MONDO disease ontology.
#keep only exact synonyms that are not deprecated, misspellings, non-human, or ambiguous.

#Below heuristics are used to filter out synonyms that are not true alternate names for the disease mention.
#1. keep synonyms that are at least 3 characters long and not the same as the disease mention itself.
#2. MONDO sometimes tags ordinary lowercase words as ABBREVIATION synonyms (e.g. "cold" for COPD. Drop those plus any uppercase sibling sharing the same casefolded text, since matching is case-insensitive.
#3. OMIM titles narrower/related entries (susceptibility loci, digenic/polygenic/somatic-mutation findings) as if they were plain synonyms of the disease. Drop synonyms matching that pattern since they're a different, narrower concept, not a true alternate name.

mondo = pronto.Ontology("data/mondo.owl")
import re

OMIM_SUSCEPTIBILITY_RE = re.compile(
    r"susceptibility to|quantitative trait locus|,\s*digenic\b|,\s*polygenic\b|,\s*somatic\b"
    r"|,\s*association with\b|,\s*modifier(?:\s+of)?\b",
    flags=re.IGNORECASE,
)

mention_synonyms = {}
for mention in unique_entities:
    for term in mondo.terms():
        if term.obsolete or term.name is None:
            continue
        synonyms = set()
        duplicate_synonyms = []
        for synonym in term.synonyms:
            if synonym.scope == "EXACT":
                if synonym.type is not None:
                    if synonym.type.id.split('#')[-1] in ["DEPRECATED", "MISSPELLING", "NON_HUMAN", "AMBIGUOUS"]:
                        continue
                    if synonym.type.id.split('#')[-1] == "ABBREVIATION" and synonym.description.islower():
                        duplicate_synonyms.append(synonym.description)
                        continue
                synonyms.add(synonym.description)
        synonyms_to_remove = set()
        if duplicate_synonyms:
            for ds in duplicate_synonyms:
                for s in synonyms:
                    if s.lower() == ds:
                        synonyms_to_remove.add(s)
            synonyms.difference_update(synonyms_to_remove)
        terms_to_check = set([term.name]).union(synonyms)
        if mention.lower() in [t.lower() for t in terms_to_check]:
            #remove mention from terms_to_check to avoid printing it as a synonym of itself
            mention_synonyms[mention] = [t for t in terms_to_check if t.lower() != mention.lower() 
                                                            and len(t)>=3
                                                            and mention.lower()+" (disease)" != t.lower()
                                                            and not OMIM_SUSCEPTIBILITY_RE.search(t)]
json.dump(mention_synonyms, open("data/disease_mention_synonyms.json", "w"), indent=2, ensure_ascii=False)


In [ ]:
mention_synonyms = json.load(open("data/disease_mention_synonyms.json", "r"))
print(len(mention_synonyms), "disease mentions with synonyms found in MONDO ontology")
with open("data/medqa_mcq_with_disease_mentions.json", "r") as f:
    orig_data = pd.DataFrame([json.loads(line) for line in f])
orig_data["disease_mentions"] = orig_data["disease_mentions"].apply(json.loads)
tr_data = pd.DataFrame(columns=["oid", "did", "tid", "prompt", "predicted_answer", "gold_answer", "disease_mention", "swapped_synonym"])

for i, row in orig_data.iterrows():
    disease_mentions = row["disease_mentions"]
    for j, mention in enumerate(disease_mentions):
        if mention in mention_synonyms:
            synonyms = mention_synonyms[mention]
            synonyms_dist = [jaccard_distance(set(mention), set(synonym)) for synonym in synonyms]
            #select synonyms with distance >= 0.25 to avoid synonyms that are too similar to the mention itself
            selected_synonyms = [synonym for synonym, dist in zip(synonyms, synonyms_dist) if (dist >= 0.25)]
            if selected_synonyms:
                for k, synonym in enumerate(selected_synonyms):
                    tr_data.loc[len(tr_data)] = {"oid": row["index"], "did": j, "tid": k, "prompt": row["prompt"].replace(mention, synonym), "predicted_answer": None, "gold_answer": row["gold_answer"], "disease_mention": mention, "swapped_synonym": synonym}
tr_data.to_json("data/tr_medqa_mcq_with_disease_mentions.json", orient="records", lines=True)